# Surface water from SAR — CSA adaptation
Role: `demostracion_registro_corto`.
Adapted raster reading, masked arrays and CRS inspection from [CSA tutorial](https://github.com/asc-csa/Synthetic_Aperture_Radar_Tutorial), Option 2 Python, BC Wildfire raster plot. MIT; copyright Canadian Space Agency 2022; see LICENSE-CSA.txt. Water thresholding is an extension, not a CSA-validated classifier.

Install numpy, rasterio, matplotlib. Obtain RCM through EODMS or Sentinel-1 through ASF/Earthdata (catalog in data/recursos/csa). Never process the tutorial preview image as calibrated backscatter. In SNAP apply orbit, thermal/border-noise removal, radiometric calibration, speckle filtering, terrain correction and convert sigma0 to dB. Fix polarization, relative orbit, incidence range, pixel spacing, AOI and valid coverage across dates. Reproject to EPSG:6933, clip to a fixed lake/wetland polygon with nodata outside. Wind roughening, flooded vegetation, shadow and wet soil can bias the mask; validate against independent outlines. -17 dB is illustrative and must be calibrated locally.

Create a manifest JSON list with date (ISO day), path (calibrated clipped GeoTIFF), source, product_id and polarization. Set SAR_MANIFEST to its path. Without actual imagery the notebook reports blocked and does not fabricate areas.

In [ ]:
import os, sys, json, csv
from pathlib import Path
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'backend/clima_nic').exists())
sys.path.insert(0, str(root/'backend'))
from clima_nic.fuentes.sar_agua import procesar
manifest = os.environ.get('SAR_MANIFEST')
rows = []
if not manifest:
    print('BLOCKED: set SAR_MANIFEST to real calibrated imagery manifest; no measured area produced.')
else:
    items = json.loads(Path(manifest).read_text())
    output = Path(os.environ.get('SAR_OUTPUT', 'sar-output'))
    output.mkdir(parents=True, exist_ok=True)
    for item in items:
        for key in ['date','path','source','product_id','polarization']:
            if not item.get(key): raise ValueError('Missing '+key)
        mask = output / (item['product_id']+'_water.tif')
        rows.append({**item, **procesar(item['path'], mask), 'mask':str(mask)})
    if rows:
        with (output/'water_area.csv').open('w', newline='') as f:
            writer=csv.DictWriter(f, fieldnames=rows[0].keys()); writer.writeheader(); writer.writerows(rows)
    print(rows)
